# 📊 Exploratory Data Analysis (EDA) - Customer Satisfaction Rating
This notebook analyzes the **Customer Support Ticket Dataset** to understand customer satisfaction patterns, missing values, and key feature relationships before model building.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load the Dataset
Loading `customer_support_tickets.csv` and inspecting basic structure.

In [2]:
df = pd.read_csv('customer_support_tickets.csv')
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

Dataset Dimensions: 8469 rows, 17 columns


## 2. Dataset Overview & Data Types
Inspecting column types, counts, and non-null values.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   str    
 2   Customer Email                8469 non-null   str    
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   str    
 5   Product Purchased             8469 non-null   str    
 6   Date of Purchase              8469 non-null   str    
 7   Ticket Type                   8469 non-null   str    
 8   Ticket Subject                8469 non-null   str    
 9   Ticket Description            8469 non-null   str    
 10  Ticket Status                 8469 non-null   str    
 11  Resolution                    2769 non-null   str    
 12  Ticket Priority               8469 non-null   str    
 13  Ticket Channel

## 3. Missing Value Analysis & Ticket Status
Notice that `Customer Satisfaction Rating`, `Resolution`, and `Time to Resolution` are only populated for **Closed** tickets (when customer feedback is collected).

In [4]:
print("--- Columns with Missing Values ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n--- Ticket Status vs Satisfaction Rating Presence ---")
print(pd.crosstab(df['Ticket Status'], df['Customer Satisfaction Rating'].notnull()))

--- Columns with Missing Values ---
Resolution                      5700
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64

--- Ticket Status vs Satisfaction Rating Presence ---
Customer Satisfaction Rating  False  True 
Ticket Status                             
Closed                            0   2769
Open                           2819      0
Pending Customer Response      2881      0


## 4. Distribution of Customer Satisfaction Ratings (1 to 5)
Filtering for closed tickets where rating exists and checking class balance.

In [5]:
closed_tickets = df[df['Customer Satisfaction Rating'].notnull()].copy()
closed_tickets['Customer Satisfaction Rating'] = closed_tickets['Customer Satisfaction Rating'].astype(int)

counts = closed_tickets['Customer Satisfaction Rating'].value_counts().sort_index()
print("Rating Distribution:")
print(counts)

plt.figure(figsize=(8, 4))
ax = sns.countplot(data=closed_tickets, x='Customer Satisfaction Rating', hue='Customer Satisfaction Rating', palette='Blues_r', legend=False)
plt.title('Customer Satisfaction Rating Distribution (Closed Tickets)')
plt.xlabel('Satisfaction Rating (1 = Low, 5 = High)')
plt.ylabel('Ticket Count')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.show()

Rating Distribution:
Customer Satisfaction Rating
1    553
2    549
3    580
4    543
5    544
Name: count, dtype: int64


## 5. Satisfaction by Ticket Priority & Ticket Type
Evaluating whether higher priority issues or specific categories lead to lower customer satisfaction.

In [6]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.barplot(data=closed_tickets, x='Ticket Priority', y='Customer Satisfaction Rating', 
            hue='Ticket Priority', estimator=np.mean, errorbar=None, palette='mako', legend=False)
plt.title('Mean Satisfaction by Priority')
plt.ylim(1, 5)

plt.subplot(1, 2, 2)
sns.barplot(data=closed_tickets, x='Ticket Type', y='Customer Satisfaction Rating', 
            hue='Ticket Type', estimator=np.mean, errorbar=None, palette='viridis', legend=False)
plt.title('Mean Satisfaction by Ticket Type')
plt.xticks(rotation=45)
plt.ylim(1, 5)

plt.tight_layout()
plt.show()

## 6. Channel & Customer Age Analysis
Analyzing if submission channel or customer demographics influence ratings.

In [7]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(data=closed_tickets, x='Ticket Channel', y='Customer Satisfaction Rating', 
            hue='Ticket Channel', palette='Set2', legend=False)
plt.title('Satisfaction Rating by Ticket Channel')

plt.subplot(1, 2, 2)
sns.histplot(data=closed_tickets, x='Customer Age', hue='Customer Satisfaction Rating', 
             multiple='stack', bins=15, palette='Spectral')
plt.title('Customer Age Distribution by Rating')

plt.tight_layout()
plt.show()

## 7. Ticket Text Word Count Analysis
Examining whether longer, more detailed ticket descriptions correlate with dissatisfied customers.

In [8]:
closed_tickets['desc_words'] = closed_tickets['Ticket Description'].fillna('').apply(lambda x: len(x.split()))

print("Description word count statistics across ratings:")
print(closed_tickets.groupby('Customer Satisfaction Rating')['desc_words'].describe()[['mean', 'std', 'min', '50%', 'max']])

plt.figure(figsize=(8, 4))
sns.boxplot(data=closed_tickets, x='Customer Satisfaction Rating', y='desc_words', 
            hue='Customer Satisfaction Rating', palette='coolwarm', legend=False)
plt.title('Ticket Description Word Count across Ratings')
plt.xlabel('Satisfaction Rating')
plt.ylabel('Word Count')
plt.show()

Description word count statistics across ratings:
                                   mean       std   min   50%   max
Customer Satisfaction Rating                                       
1                             46.875226  7.588415  23.0  49.0  62.0
2                             46.001821  8.821941  21.0  48.0  61.0
3                             46.617241  8.292268  21.0  48.0  62.0
4                             46.421731  8.367088  21.0  49.0  61.0
5                             46.375000  8.803722  21.0  49.0  62.0


## 8. Summary of Findings
1. **Target Filter:** Customer Satisfaction Rating is present in **2,769 closed tickets** (scale 1 to 5).
2. **Balanced Target:** The 5 classes are evenly distributed (~540 to 580 instances each).
3. **Core Features for ML:**
   - **Text:** Combination of `Ticket Subject` + `Ticket Description`.
   - **Categorical:** `Ticket Type`, `Ticket Priority`, `Ticket Channel`, `Product Purchased`.
   - **Numerical:** `Customer Age`.
4. **Next Step:** Open `02_Model_Customer_Satisfaction.ipynb` to build, train, evaluate, and export the Machine Learning model.